# CNN Baseline Classification

In [ ]:
# Imports
import h5py
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configs
DS_PATH    = r'path/to/dataset.h5'   # path to your HDF5 file
BINARIZE   = False               # True = threshold pixels to match Tsetlin Machine input format (0 or 1)
IMG_SIZE   = 240                 # images are resized to IMG_SIZE × IMG_SIZE
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-3
PATIENCE   = 7                   # early-stopping patience (monitored on val MCC)
K_FOLDS    = [0, 1, 2, 3, 4]

In [ ]:
# Data helpers 
def get_train_val_data(ds_name, k_fold_index=4):
    train_val_plans = {
        4: ([0, 1, 2, 3], 4),
        3: ([0, 1, 2, 4], 3),
        2: ([0, 1, 3, 4], 2),
        1: ([0, 2, 3, 4], 1),
        0: ([1, 2, 3, 4], 0),
    }
    train_folds, val_fold = train_val_plans[k_fold_index]
    with h5py.File(ds_name, 'r') as f:
        X_train, Y_train = [], []
        for fold in train_folds:
            X_train.append(f[f'fold_{fold}']['image'][:])
            Y_train.append(f[f'fold_{fold}']['target'][:])
        X_train = np.concatenate(X_train, axis=0)
        Y_train = np.concatenate(Y_train, axis=0)
        X_val = f[f'fold_{val_fold}']['image'][:]
        Y_val = f[f'fold_{val_fold}']['target'][:]
    return (X_train, Y_train), (X_val, Y_val)


def get_test_data(ds_name):
    with h5py.File(ds_name, 'r') as f:
        X_test = f['fold_5']['image'][:]
        Y_test = f['fold_5']['target'][:]
    return X_test, Y_test


def preprocess(X, binarize=False):
    """Normalise to [0,1], optionally binarise, resize, add channel dim."""
    if X.ndim == 4 and X.shape[-1] == 1:
        X = X[..., 0]
    X = X.astype(np.float32)
    if X.max() > 1.0:
        X = X / 255.0
    if binarize:
        X = (X >= 0.5).astype(np.float32)
    if X.shape[1] != IMG_SIZE or X.shape[2] != IMG_SIZE:
        X = tf.image.resize(X[..., np.newaxis], [IMG_SIZE, IMG_SIZE]).numpy()[..., 0]
    return X[..., np.newaxis]   # (N, H, W, 1) — Keras channels-last

In [ ]:
# Model
def build_model():
    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
# Metrics
def calculate_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='binary', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='binary', zero_division=0),
        'f1':        f1_score(y_true, y_pred, average='binary', zero_division=0),
        'mcc':       matthews_corrcoef(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_prob),
    }
    cm = confusion_matrix(y_true, y_pred)
    return metrics, cm


def print_metrics(metrics, cm):
    for k, v in metrics.items():
        print(f"  {k:12s}: {v:.4f}")
    print(f"  confusion:\n{cm}")

In [ ]:
# MCC early-stopping callback 
# Keras's built-in EarlyStopping only watches loss or accuracy.
# This callback saves the weights whenever val MCC improves,
# and stops training when it hasn't improved for PATIENCE # of epochs.
class MCCEarlyStopping(keras.callbacks.Callback):
    def __init__(self, X_val, Y_val, patience=7):
        super().__init__()
        self.X_val    = X_val
        self.Y_val    = Y_val
        self.patience = patience
        self.best_mcc = -2.0
        self.wait     = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.X_val, verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)
        mcc = matthews_corrcoef(self.Y_val, y_pred)
        print(f"  val_mcc: {mcc:.4f}", end='')

        if mcc > self.best_mcc:
            self.best_mcc     = mcc
            self.best_weights = self.model.get_weights()
            self.wait         = 0
            print("  Improved")
        else:
            self.wait += 1
            print(f"  (No improvement {self.wait}/{self.patience})")
            if self.wait >= self.patience:
                print(f"  Early stop. Restoring best weights (MCC={self.best_mcc:.4f})")
                self.model.set_weights(self.best_weights)
                self.model.stop_training = True

In [ ]:
# Train all folds
fold_results = {}

for fold in K_FOLDS:
    print(f"\n{'='*55}\n  FOLD {fold}\n{'='*55}")

    (X_tr, Y_tr), (X_val, Y_val) = get_train_val_data(DS_PATH, fold)
    X_tr  = preprocess(X_tr,  binarize=BINARIZE)
    X_val = preprocess(X_val, binarize=BINARIZE)

    # Upweight the positive class to handle imbalance
    neg, pos = (Y_tr == 0).sum(), (Y_tr == 1).sum()
    class_weight = {0: 1.0, 1: neg / max(pos, 1)}
    print(f"  Train: {len(Y_tr)}  Val: {len(Y_val)}  pos class weight: {class_weight[1]:.2f}")

    model = build_model()

    history = model.fit(
        X_tr, Y_tr,
        validation_data=(X_val, Y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[MCCEarlyStopping(X_val, Y_val, patience=PATIENCE)],
        verbose=1,
    )

    # Final evaluation with best weights already restored by callback
    y_prob = model.predict(X_val, verbose=0).ravel()
    metrics, cm = calculate_metrics(Y_val.astype(int), y_prob)

    print(f"\n  ── Val metrics ──")
    print_metrics(metrics, cm)

    fold_results[fold] = {
        'model':   model,
        'metrics': metrics,
        'cm':      cm,
        'history': history.history,
    }

In [ ]:
# Cross-validation summary
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'mcc', 'roc_auc']

print("\n" + "="*55)
print("  CROSS-VALIDATION SUMMARY")
print("="*55)

per_metric = {k: [] for k in metric_keys}
for fold, res in fold_results.items():
    scores = '  '.join(f"{k}: {res['metrics'][k]:.4f}" for k in metric_keys)
    print(f"  Fold {fold}: {scores}")
    for k in metric_keys:
        per_metric[k].append(res['metrics'][k])

print("\n  Mean ± Std:")
for k, vals in per_metric.items():
    print(f"    {k:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for fold, res in fold_results.items():
    h = res['history']
    axes[0].plot(h['loss'],         label=f'train f{fold}', alpha=0.6)
    axes[0].plot(h['val_loss'],     label=f'val f{fold}',   alpha=0.6, linestyle='--')
    axes[1].plot(h['accuracy'],     label=f'train f{fold}', alpha=0.6)
    axes[1].plot(h['val_accuracy'], label=f'val f{fold}',   alpha=0.6, linestyle='--')

for ax, title in zip(axes, ['Loss', 'Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120)
plt.show()

In [ ]:
# Test-set evaluation (best fold by val MCC)
best_fold = max(fold_results, key=lambda f: fold_results[f]['metrics']['mcc'])
print(f"Best fold by val MCC: {best_fold}")

X_test, Y_test = get_test_data(DS_PATH)
X_test = preprocess(X_test, binarize=BINARIZE)

y_prob = fold_results[best_fold]['model'].predict(X_test, verbose=0).ravel()
test_metrics, test_cm = calculate_metrics(Y_test.astype(int), y_prob)

print("\n" + "="*55)
print("  TEST SET METRICS")
print("="*55)
print_metrics(test_metrics, test_cm)